# ***CLASIFICADOR BASELINE (ResNet50)****

## Imports y rutas

 ## Librerías

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

## rutas

In [ ]:
# defino todas las rutas apuntando al dataset privado con los PNGs ya convertidos
RUTA_DATOS      = '/kaggle/input/competitions/rsna-pneumonia-detection-challenge'
RUTA_PNG_TRAIN  = '/kaggle/input/datasets/franciscofdzfer/rsna-png-512/png_train'
RUTA_PNG_TEST   = '/kaggle/input/datasets/franciscofdzfer/rsna-png-512-test/png_test'
RUTA_OUTPUTS    = '/kaggle/working'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')

## Cargar labels

In [ ]:
# cargo el CSV y elimino duplicados para tener una sola fila por paciente con su target
# lee las etiquetas y limpia los datos
df_labels = pd.read_csv(f'{RUTA_DATOS}/stage_2_train_labels.csv')
# elimina duplicados para tener pacientes únicos
df = df_labels.drop_duplicates(subset='patientId')[['patientId', 'Target']]
# crea la columna con rutas PNG
df['ruta_png'] = df['patientId'].apply(lambda x: f'{RUTA_PNG_TRAIN}/{x}.png')

df = df.sample(n=2000, random_state=42)

print(df.shape)
print(df['Target'].value_counts())

## split

In [ ]:
# divido manteniendo la proporción de clases para que val sea representativo del train
df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['Target']
)

print(f'train: {len(df_train)} — positivos: {df_train["Target"].sum()}')
print(f'val:   {len(df_val)}   — positivos: {df_val["Target"].sum()}')

## Dataset class PyTorch

In [ ]:
# defino la clase que carga cada PNG y devuelve imagen + etiqueta lista para el modelo
class RSNADataset(Dataset):
    # # guarda la tabla y transformaciones iniciales
    def __init__(self, df, transform=None):
        # # reinicia los índices para evitar errores
        self.df        = df.reset_index(drop=True)
        # almacena las funciones de aumento de datos
        self.transform = transform

    # calcula el total de pacientes registrados
    def __len__(self):
        return len(self.df)

    # extrae un paciente específico usando su índice
    def __getitem__(self, idx):
        # busca la ruta de la imagen png
        ruta   = self.df.loc[idx, 'ruta_png']
        # obtiene si tiene neumonía o no
        target = self.df.loc[idx, 'Target']
        # abre la imagen asegurando los tres canales
        img    = Image.open(ruta).convert('RGB')
        # aplica los cambios si están definidos
        if self.transform:
            # transforma la imagen para el modelo
            img = self.transform(img)
        # entrega los datos listos para PyTorch
        return img, torch.tensor(target, dtype=torch.float32)

## Transforms y DataLoaders

In [ ]:
# preparo las imágenes para el modelo las redimensiono, normalizo y
# organizo en lotes para entrenar y validar.

# agrupa varias transformaciones para el entrenamiento
transform_train = transforms.Compose([
    # reduce la imagen a 224 píxeles
    transforms.Resize((224, 224)),
    # voltea horizontalmente de forma aleatoria la radiografía
    transforms.RandomHorizontalFlip(),
    # altera el brillo y contraste aleatoriamente
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    # convierte la imagen en un tensor numérico
    transforms.ToTensor(),
    # normaliza los canales usando valores estándar
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# agrupa las transformaciones para la validación
transform_val = transforms.Compose([
    # cambia el tamaño igual que en entrenamiento
    transforms.Resize((224, 224)),
    # transforma la estructura a tensor matemático
    transforms.ToTensor(),
    # normaliza con idénticos valores de escala
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# crea el dataset con datos de entrenamiento
ds_train = RSNADataset(df_train, transform=transform_train)
# genera el dataset para validar el modelo
ds_val   = RSNADataset(df_val,   transform=transform_val)

# prepara lotes mezclados para el entrenamiento
dl_train = DataLoader(ds_train, batch_size=32, shuffle=True,  num_workers=2)
# crea lotes ordenados para la validación
dl_val   = DataLoader(ds_val,   batch_size=32, shuffle=False, num_workers=2)

# muestra la cantidad de lotes de entrenamiento
print(f'batches train: {len(dl_train)}')
# imprime los lotes totales de validación
print(f'batches val:   {len(dl_val)}')

## Modelo ResNet50

In [ ]:
# cargo ResNet50 preentrenado, congelo sus capas y adapto la última para detectar neumonía
def crear_modelo():
    # descarga una red resnet50 preentrenada
    modelo = models.resnet50(weights='IMAGENET1K_V1')
    # congela todas las capas para no modificarlas
    for param in modelo.parameters():
        # desactiva el cálculo de gradientes del modelo
        # *********    SI ESTA A TRUE TODAS LAS CAPAS APRENDEN *******
        param.requires_grad = False
    # reemplaza la última capa para clasificar y DECIDIR SI HAY O NO NEUMONIA
    modelo.fc = nn.Linear(2048, 1)
    # verifica si hay múltiples tarjetas gráficas disponibles
    if torch.cuda.device_count() > 1:
        # avisa cuántas unidades de procesamiento detecta
        print(f'usando {torch.cuda.device_count()} GPUs')
        # adapta el modelo para entrenar en paralelo
        modelo = nn.DataParallel(modelo)
    # envía el modelo completo al hardware seleccionado
    return modelo.to(DEVICE)

# inicializa el modelo ejecutando la función creada
modelo = crear_modelo()
# muestra la arquitectura completa de la red
print(modelo)

## Loss y optimizer

In [ ]:
# defino la loss con peso para compensar el desbalance de clases y el optimizer solo sobre la última capa
# cuenta cuántos pacientes no tienen neumonía
n_neg = (df_train['Target'] == 0).sum()
# cuenta cuántos pacientes sí tienen neumonía
n_pos = (df_train['Target'] == 1).sum()
# calcula el balance de peso entre clases
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)

# define la función de pérdida con peso
criterio   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# configura el optimizador adam con tasa fija
optimizer  = torch.optim.Adam(modelo.parameters(), lr=1e-4)
# reduce la tasa si el aprendizaje estanca
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)

# el valor del peso de compensación
print(f'pos_weight: {pos_weight.item():.3f}')

## Funciones de entrenamiento y evaluación

In [ ]:
# separo train y eval en funciones cortas para mantener el bucle principal limpio y legible

# función de entrenamiento por época
def train_epoch(modelo, dl, criterio, optimizer):
    # activa el modo entrenamiento del modelo
    modelo.train()
    # inicializa el acumulador de la pérdida
    perdida_total = 0
    # recorre los lotes de imágenes y etiquetas
    for imgs, labels in dl:
        # envía los datos al hardware configurado
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        # limpia los gradientes del paso anterior
        optimizer.zero_grad()
        # calcula predicciones y ajusta sus dimensiones
        preds = modelo(imgs).squeeze(1)
        # calcula el error del lote actual
        loss  = criterio(preds, labels)
        # realiza la retropropagación de los errores
        loss.backward()
        # actualiza los pesos de la red
        optimizer.step()
        # acumula la pérdida obtenida en lote
        perdida_total += loss.item()
    # promedio general de pérdida
    return perdida_total / len(dl)

# función de evaluación por época
def eval_epoch(modelo, dl, criterio):
    # activa el modo evaluación del modelo
    modelo.eval()
    # inicializa el acumulador de pérdida total
    perdida_total = 0
    # crea listas vacías para métricas finales
    todos_preds, todos_labels = [], []
    # desactiva el cálculo de los gradientes
    with torch.no_grad():
        # recorre el cargador de datos
        for imgs, labels in dl:
            # envía imágenes y etiquetas al dispositivo
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            # genera predicciones sin modificar la red
            preds = modelo(imgs).squeeze(1)
            # calcula y acumula el error obtenido
            perdida_total += criterio(preds, labels).item()
            # aplica sigmoide y almacena las predicciones
            todos_preds.extend(torch.sigmoid(preds).cpu().numpy())
            # guarda las etiquetas reales del lote
            todos_labels.extend(labels.cpu().numpy())
    # calcula el puntaje auc de clasificación
    auc = roc_auc_score(todos_labels, todos_preds)
    # pérdida promedio y auc
    return perdida_total / len(dl), auc

## Bucle de entrenamiento

In [ ]:
# entreno con early stopping guardando el mejor modelo según el AUC de validación

# cantidad máxima de epocas totales
EPOCHS       = 20
# límite de épocas sin mejora
patience     = 3
# inicializa el mejor puntaje de validación
mejor_auc    = 0
# contador para activar el frenado temprano
epocas_sin_mejora = 0
# ruta para guardar pesos finales
ruta_modelo  = f'{RUTA_OUTPUTS}/mejor_resnet50.pth'

# inicia el bucle para entrenar por épocas
for epoch in range(EPOCHS):
    # entrena el modelo y obtiene el error
    loss_train          = train_epoch(modelo, dl_train, criterio, optimizer)
    # evalúa el modelo calculando error y auc
    loss_val, auc_val   = eval_epoch(modelo, dl_val, criterio)
    # ajusta la tasa según el error obtenido
    scheduler.step(loss_val)

    # imprime el rendimiento resumido de la época
    print(f'epoch {epoch+1:02d} — loss_train: {loss_train:.4f} — loss_val: {loss_val:.4f} — auc_val: {auc_val:.4f}')

    # comprueba si el modelo actual es superior
    if auc_val > mejor_auc:
        # actualiza la marca del mejor puntaje registrado
        mejor_auc = auc_val
        # guarda los mejores pesos en el disco
        torch.save(modelo.state_dict(), ruta_modelo)
        # guardado mostrando el nuevo auc
        print(f'  ✓ mejor modelo guardado (auc: {mejor_auc:.4f})')
        # reinicia el contador de paciencia a cero
        epocas_sin_mejora = 0
    else:
        # incrementa el contador de épocas sin progreso
        epocas_sin_mejora += 1
        # verifica si se alcanzó la tolerancia establecida
        if epocas_sin_mejora >= patience:
            # cierre anticipado del entrenamiento
            print(f'early stopping en epoch {epoch+1}')
            break

## Curva ROC y matriz de confusión

In [ ]:
# evalúo el mejor modelo sobre validación para entender dónde falla antes de generar la submission

# carga los mejores pesos guardados en disco
modelo.load_state_dict(torch.load(ruta_modelo))
# evalúa el modelo cargado obteniendo su auc
_, auc_val = eval_epoch(modelo, dl_val, criterio)

# recojo predicciones
# activa el modo de evaluación del modelo
modelo.eval()
# crea listas vacías para guardar las salidas
todos_preds, todos_labels = [], []
# deshabilita el cálculo de gradientes en PyTorch
with torch.no_grad():
    # recorre el cargador con datos de validación
    for imgs, labels in dl_val:
        # envía las imágenes al hardware de procesamiento
        imgs   = imgs.to(DEVICE)
        # predice aplicando sigmoide para obtener probabilidades
        preds  = torch.sigmoid(modelo(imgs).squeeze(1))
        # almacena las probabilidades en la lista general
        todos_preds.extend(preds.cpu().numpy())
        # guarda las etiquetas verdaderas como arreglos numpy
        todos_labels.extend(labels.numpy())

# curva ROC
# calcula los componentes de la curva roc
fpr, tpr, umbrales = roc_curve(todos_labels, todos_preds)
# crea una ventana con dos gráficos alineados
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(fpr, tpr, label=f'AUC = {auc_val:.4f}')
axes[0].plot([0,1],[0,1], 'k--')
axes[0].set_title('curva ROC')
axes[0].legend()

# matriz de confusión con umbral 0.5
# binariza las predicciones usando el umbral medio
preds_bin = [1 if p > 0.5 else 0 for p in todos_preds]
# genera la matriz comparando datos reales y predichos
cm = confusion_matrix(todos_labels, preds_bin)
axes[1].imshow(cm, cmap='Blues')
axes[1].set_title('matriz de confusión (umbral 0.5)')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, cm[i,j], ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.show()

## Inferencia sobre test y submission

In [ ]:
# cargo las imágenes de test, genero predicciones y construyo el CSV en el formato requerido por Kaggle
# extrae los identificadores quitando la extensión png
ids_test = [f.replace('.png', '') for f in os.listdir(RUTA_PNG_TEST)]

# reutiliza las transformaciones del conjunto de validación
transform_test = transform_val

# inicializa una lista vacía para almacenar predicciones
resultados = []
# activa el modo de evaluación del modelo
modelo.eval()
# deshabilita el cálculo de gradientes en PyTorch
with torch.no_grad():
    # recorre cada identificador de paciente del test
    for pid in ids_test:
        # construye la ruta de acceso al archivo png
        ruta_png = f'{RUTA_PNG_TEST}/{pid}.png'
        # abre la imagen convirtiéndola a tres canales rgb
        img      = Image.open(ruta_png).convert('RGB')
        # transforma la imagen y añade dimensión de lote
        img      = transform_test(img).unsqueeze(0).to(DEVICE)
        # calcula la probabilidad final usando la función sigmoide
        prob     = torch.sigmoid(modelo(img)).item()

        # evalúa si la probabilidad supera el umbral establecido
        if prob > 0.5:
            # bbox de imagen completa como proxy
            # asigna la caja completa simulando una detección
            pred_str = f'{prob:.4f} 0 0 1024 1024'
        else:
            # define una cadena vacía al no detectar neumonía
            pred_str = ''

        # agrega el diccionario con datos a la lista
        resultados.append({'patientId': pid, 'PredictionString': pred_str})

# convierte la lista de resultados en una tabla pandas
df_sub = pd.DataFrame(resultados)
# guarda la tabla en disco como archivo csv formal
df_sub.to_csv(f'{RUTA_OUTPUTS}/submission_resnet50.csv', index=False)
# tamaño total del archivo de sumisión creado
print(f'submission generada: {len(df_sub)} filas')
df_sub.head()

In [ ]:
El modelo clasifica cada radiografía como neumonía o no neumonía.

Si la probabilidad superaba 0.5 marcaba toda la imagen como zona enferma.

La métrica exige localizar el foco exacto, no la imagen completa.

# *** FASE 2.1 ***

# Imports y rutas

In [ ]:
# importo las librerías necesarias para todo el notebook
import os
import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

# defino las rutas base del entorno kaggle
RUTA_DATOS   = '/kaggle/input/competitions/rsna-pneumonia-detection-challenge'
# carpeta con las radiografías en formato dicom
RUTA_DICOM   = f'{RUTA_DATOS}/stage_2_train_images'
# carpeta con las radiografías de test
RUTA_TEST    = f'{RUTA_DATOS}/stage_2_test_images'
# carpeta de salida para modelos y submissions
RUTA_OUTPUTS = '/kaggle/working'

# detecta si hay gpu disponible y la asigna
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')

# Cargar labels

In [ ]:
# cargo el archivo principal, elimina los registros duplicados de pacientes y selecciona una muestra
# de 2000 casos para entrenar.

# cargo el csv principal y construyo un df limpio con una fila por paciente
df_labels = pd.read_csv(f'{RUTA_DATOS}/stage_2_train_labels.csv')
# elimino duplicados para tener un registro único por paciente
df = df_labels.drop_duplicates(subset='patientId')[['patientId', 'Target']]
# selecciono 2000 imágenes al azar para el entrenamiento
df = df.sample(n=2000, random_state=42)
# muestro el tamaño y distribución de clases
print(df.shape)
# cuento cuántos pacientes hay por clase
print(df['Target'].value_counts())

# Dataset class DICOM

In [ ]:
# estructura que lee archivos médicos DICOM, normaliza en tres canales visuales
# y los transforma en tensores utilizables

# defino la clase que lee el dicom y lo convierte a tensor en memoria
class RSNADataset(Dataset):
    # inicializa el dataset con el dataframe y las transformaciones
    def __init__(self, df, transform=None):
        # reinicia los índices para evitar errores de acceso
        self.df        = df.reset_index(drop=True)
        # almacena las transformaciones a aplicar
        self.transform = transform

    # devuelve el número total de muestras
    def __len__(self):
        return len(self.df)

    # carga y procesa una muestra dado su índice
    def __getitem__(self, idx):
        # obtiene el identificador del paciente
        pid    = self.df.loc[idx, 'patientId']
        # obtiene la etiqueta de neumonía
        target = self.df.loc[idx, 'Target']
        # lee el archivo dicom
        dcm    = pydicom.dcmread(f'{RUTA_DICOM}/{pid}.dcm')
        # extrae el array de píxeles como float
        img    = dcm.pixel_array.astype(np.float32)
        # invierte si la imagen está en monochrome1
        if dcm.PhotometricInterpretation == 'MONOCHROME1':
            img = img.max() - img
        # normaliza los valores al rango 0-1
        img = (img - img.min()) / (img.max() - img.min() + 1e-6)
        # replica el canal gris a 3 canales para imagenet
        img = np.stack([img, img, img], axis=2)
        # convierte el array a imagen PIL para aplicar transforms
        img = Image.fromarray((img * 255).astype(np.uint8))
        # aplica las transformaciones si están definidas
        if self.transform:
            img = self.transform(img)
        # devuelve imagen y etiqueta como tensores
        return img, torch.tensor(target, dtype=torch.float32)

# Split y DataLoaders

In [ ]:
# divido el dataset manteniendo la proporción de clases entre train y val
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Target'])

# preparo y normalizo numéricamente las radiografías para procesar con idéntico tamaño
transform_train = transforms.Compose([
    # redimensiono a 224 para resnet50
    transforms.Resize((224, 224)),
    # volteo horizontal aleatorio para aumentar variedad
    transforms.RandomHorizontalFlip(),
    # ajuste aleatorio de brillo y contraste
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    # convierto a tensor normalizado
    transforms.ToTensor(),
    # normalizo con valores estándar de imagenet
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# preparo y normalizo numéricamente las radiografías para procesar con idéntico tamaño
transform_val = transforms.Compose([
    # redimensiono igual que en train
    transforms.Resize((224, 224)),
    # convierto a tensor
    transforms.ToTensor(),
    # normalizo con los mismos valores
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# creo los datasets y dataloaders
ds_train = RSNADataset(df_train, transform=transform_train)
ds_val   = RSNADataset(df_val,   transform=transform_val)
dl_train = DataLoader(ds_train, batch_size=32, shuffle=True,  num_workers=4)
dl_val   = DataLoader(ds_val,   batch_size=32, shuffle=False, num_workers=4)

# muestro el número de lotes por split
print(f'batches train: {len(dl_train)}')
print(f'batches val:   {len(dl_val)}')

# Funciones de entrenamiento y evaluación

In [ ]:
# entreno procesando por grupos, calcula los errores
# ajusta los pesos matemáticos y devuelve la pérdida promedio fina
def train_epoch(modelo, dl, criterio, optimizer):
    # activa el modo entrenamiento del modelo
    modelo.train()
    # inicializa el acumulador de pérdida
    perdida_total = 0
    # recorre los lotes de imágenes y etiquetas
    for imgs, labels in dl:
        # envía los datos al dispositivo configurado
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        # limpia los gradientes del paso anterior
        optimizer.zero_grad()
        # calcula las predicciones del modelo
        preds = modelo(imgs).squeeze(1)
        # calcula el error del lote actual
        loss  = criterio(preds, labels)
        # realiza la retropropagación
        loss.backward()
        # actualiza los pesos de la red
        optimizer.step()
        # acumula la pérdida del lote
        perdida_total += loss.item()
    # devuelve la pérdida media por lote
    return perdida_total / len(dl)

# evalúo el modelo congelado con datos de validación, calculo el error promedio
# y mido la precisión diagnóstica mediante AUC
def eval_epoch(modelo, dl, criterio):
    # activa el modo evaluación del modelo
    modelo.eval()
    # inicializa el acumulador de pérdida
    perdida_total = 0
    # crea listas para guardar predicciones y etiquetas
    todos_preds, todos_labels = [], []
    # deshabilita el cálculo de gradientes
    with torch.no_grad():
        # recorre los lotes de validación
        for imgs, labels in dl:
            # envía los datos al dispositivo
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            # genera predicciones sin modificar la red
            preds = modelo(imgs).squeeze(1)
            # acumula la pérdida del lote
            perdida_total += criterio(preds, labels).item()
            # aplica sigmoide y almacena las predicciones
            todos_preds.extend(torch.sigmoid(preds).cpu().numpy())
            # almacena las etiquetas reales
            todos_labels.extend(labels.cpu().numpy())
    # calcula el auc sobre todas las predicciones
    auc = roc_auc_score(todos_labels, todos_preds)
    # devuelve pérdida media y auc
    return perdida_total / len(dl), auc

# modelo

In [ ]:
# modelo preentrenado ResNet50, activa el entrenamiento paralelo en ambas tarjetas gráficas
# y ajusta el peso para clases desequilibradas

# fase a — entreno solo la última capa para que aprenda la tarea antes de descongelar
modelo = models.resnet50(weights='IMAGENET1K_V1')
# congela todas las capas del backbone
for param in modelo.parameters():
    param.requires_grad = False
# reemplaza la última capa para clasificación binaria
modelo.fc = nn.Linear(2048, 1)
# activa los gradientes solo en la última capa
for param in modelo.fc.parameters():
    param.requires_grad = True
# adapta el modelo para 2 gpus si están disponibles
if torch.cuda.device_count() > 1:
    print(f'usando {torch.cuda.device_count()} GPUs')
    modelo = nn.DataParallel(modelo)
modelo = modelo.to(DEVICE)

# Calcula la proporción entre sanos y enfermos para equilibrar los errores, define el optimizador
# reduce el aprendizaje si estanca
n_neg      = (df_train['Target'] == 0).sum()
n_pos      = (df_train['Target'] == 1).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
criterio   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.Adam(modelo.parameters(), lr=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)

print(f'pos_weight: {pos_weight.item():.3f}')

# Fase A: bucle de entrenamiento con backbone congelado

In [ ]:
# entreno solo la última capa durante 10 epochs para que aprenda la tarea básica
EPOCHS_A          = 10
patience          = 3
mejor_auc_a       = 0
epocas_sin_mejora = 0
ruta_modelo_a     = f'{RUTA_OUTPUTS}/modelo_fase_a.pth'

for epoch in range(EPOCHS_A):
    # entrena una época y obtiene la pérdida
    loss_train        = train_epoch(modelo, dl_train, criterio, optimizer)
    # evalúa el modelo y obtiene pérdida y auc
    loss_val, auc_val = eval_epoch(modelo, dl_val, criterio)
    # ajusta el lr según la pérdida de validación
    scheduler.step(loss_val)
    # muestra el resumen de la época
    print(f'[fase a] epoch {epoch+1:02d} — loss_train: {loss_train:.4f} — loss_val: {loss_val:.4f} — auc_val: {auc_val:.4f}')
    # comprueba si el modelo ha mejorado
    if auc_val > mejor_auc_a:
        # actualiza el mejor auc registrado
        mejor_auc_a = auc_val
        # guarda los pesos del mejor modelo
        torch.save(modelo.state_dict(), ruta_modelo_a)
        # informa del nuevo mejor modelo
        print(f'  ✓ mejor modelo fase a guardado (auc: {mejor_auc_a:.4f})')
        # reinicia el contador de paciencia
        epocas_sin_mejora = 0
    else:
        # incrementa el contador de épocas sin mejora
        epocas_sin_mejora += 1
        # verifica si se alcanzó el límite de paciencia
        if epocas_sin_mejora >= patience:
            # detiene el entrenamiento anticipadamente
            print(f'early stopping fase a en epoch {epoch+1}')
            break

print(f'\nmejor auc fase a: {mejor_auc_a:.4f}')

# Fase B: descongelar backbone y continuar entrenamiento

In [ ]:
# cargo el mejor modelo previo, libera todas sus capas para ajustarlas despacio a ritmo lento
# y aplica parada temprana por estancamiento

# cargo los mejores pesos de fase a
modelo.load_state_dict(torch.load(ruta_modelo_a))

# descongelo todas las capas del backbone
for param in modelo.parameters():
    # activa los gradientes en todas las capas
    param.requires_grad = True

# lr muy bajo para no destruir lo aprendido en fase a
optimizer_b = torch.optim.Adam(modelo.parameters(), lr=1e-5)
scheduler_b = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_b, patience=2)

# variables para el entrenamiento de fase b
EPOCHS_B          = 20
mejor_auc_b       = 0
epocas_sin_mejora = 0
ruta_modelo_b     = f'{RUTA_OUTPUTS}/modelo_fase_b.pth'

for epoch in range(EPOCHS_B):
    # entrena una época con backbone descongelado
    loss_train        = train_epoch(modelo, dl_train, criterio, optimizer_b)
    # evalúa el modelo y obtiene pérdida y auc
    loss_val, auc_val = eval_epoch(modelo, dl_val, criterio)
    # ajusta el lr según la pérdida de validación
    scheduler_b.step(loss_val)
    # muestra el resumen de la época
    print(f'[fase b] epoch {epoch+1:02d} — loss_train: {loss_train:.4f} — loss_val: {loss_val:.4f} — auc_val: {auc_val:.4f}')
    # comprueba si el modelo ha mejorado
    if auc_val > mejor_auc_b:
        # actualiza el mejor auc registrado
        mejor_auc_b = auc_val
        # guarda los pesos del mejor modelo
        torch.save(modelo.state_dict(), ruta_modelo_b)
        # informa del nuevo mejor modelo
        print(f'  ✓ mejor modelo fase b guardado (auc: {mejor_auc_b:.4f})')
        # reinicia el contador de paciencia
        epocas_sin_mejora = 0
    else:
        # incrementa el contador de épocas sin mejora
        epocas_sin_mejora += 1
        # verifica si se alcanzó el límite de paciencia
        if epocas_sin_mejora >= patience:
            # detiene el entrenamiento anticipadamente
            print(f'early stopping fase b en epoch {epoch+1}')
            break

print(f'\nmejor auc fase a: {mejor_auc_a:.4f}')
print(f'mejor auc fase b: {mejor_auc_b:.4f}')

# Evaluación: curva ROC y matriz de confusión

In [ ]:
# evalúo el mejor modelo de fase b para entender dónde falla antes de generar la submission
modelo.load_state_dict(torch.load(ruta_modelo_b))
_, auc_val = eval_epoch(modelo, dl_val, criterio)

modelo.eval()
todos_preds, todos_labels = [], []
with torch.no_grad():
    for imgs, labels in dl_val:
        imgs   = imgs.to(DEVICE)
        preds  = torch.sigmoid(modelo(imgs).squeeze(1))
        todos_preds.extend(preds.cpu().numpy())
        todos_labels.extend(labels.numpy())

fpr, tpr, _ = roc_curve(todos_labels, todos_preds)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, label=f'AUC = {auc_val:.4f}')
axes[0].plot([0,1],[0,1], 'k--')
axes[0].set_title('curva ROC — fase b')
axes[0].legend()

preds_bin = [1 if p > 0.5 else 0 for p in todos_preds]
cm = confusion_matrix(todos_labels, preds_bin)
axes[1].imshow(cm, cmap='Blues')
axes[1].set_title('matriz de confusión (umbral 0.5)')

# etiquetas de filas y columnas
etiquetas = ['Negativo', 'Positivo']
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Pred Negativo', 'Pred Positivo'], fontsize=10)
axes[1].set_yticklabels(['Real Negativo', 'Real Positivo'], fontsize=10)
axes[1].set_xlabel('Predicción', fontsize=11)
axes[1].set_ylabel('Real', fontsize=11)

# nombres de cada celda
nombres = [['Verdadero\nNegativo', 'Falso\nPositivo'],
           ['Falso\nNegativo',     'Verdadero\nPositivo']]

for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{nombres[i][j]}\n{cm[i,j]}',
                    ha='center', va='center', fontsize=11,
                    color='white' if cm[i,j] > cm.max()/2 else 'black')

plt.tight_layout()
plt.show()

# Instalar e importar GradCAM

In [ ]:
# instalo gradcam para generar mapas de activación sobre el modelo entrenado
!pip install grad-cam --quiet

# importo las clases necesarias para gradcam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Configurar GradCAM

In [ ]:
# configuro gradcam sobre la última capa convolucional para localizar el foco de neumonía
# extrae el modelo base sin el wrapper de dataparallel
modelo_base   = modelo.module if hasattr(modelo, 'module') else modelo
# apunta a la última capa convolucional de resnet50
capa_objetivo = [modelo_base.layer4[-1]]
# activa los gradientes necesarios para gradcam
for param in modelo_base.parameters():
    # habilita el cálculo de gradientes en cada parámetro
    param.requires_grad = True
# inicializa gradcam con el modelo y la capa objetivo
cam = GradCAM(model=modelo_base, target_layers=capa_objetivo)

#  Función para extraer bbox desde GradCAM

In [ ]:
# convierto el mapa de calor en un bbox rectangular tomando la región con mayor activación
def extraer_bbox(mapa_cam, umbral=0.5, tam_original=1024):
    # binarizo el mapa por encima del umbral establecido
    mapa_bin = (mapa_cam >= umbral).astype(np.uint8)
    # busco las filas con activación positiva
    filas = np.any(mapa_bin, axis=1)
    # busco las columnas con activación positiva
    cols  = np.any(mapa_bin, axis=0)
    # si no hay activación devuelvo none
    if not filas.any():
        return None
    # obtengo los límites verticales del área activa
    y_min, y_max = np.where(filas)[0][[0, -1]]
    # obtengo los límites horizontales del área activa
    x_min, x_max = np.where(cols)[0][[0, -1]]
    # calculo el factor de escala de 224 al tamaño original
    escala = tam_original / mapa_cam.shape[0]
    # escalo las coordenadas al tamaño original de la imagen
    x = int(x_min * escala)
    y = int(y_min * escala)
    w = int((x_max - x_min) * escala)
    h = int((y_max - y_min) * escala)
    # devuelvo las coordenadas del bbox
    return x, y, w, h

# Visualizar GradCAM sobre ejemplos positivos

In [ ]:
# muestro el mapa de calor sobre 3 casos positivos para verificar que localiza bien el foco
ids_positivos = df_val[df_val['Target'] == 1]['patientId'].values[:3]
# creo la figura con 3 filas y 2 columnas
fig, axes = plt.subplots(3, 2, figsize=(10, 14))

for idx, pid in enumerate(ids_positivos):
    # leo el dicom directamente para la visualización
    dcm    = pydicom.dcmread(f'{RUTA_DICOM}/{pid}.dcm')
    img    = dcm.pixel_array.astype(np.float32)
    # invierto si es monochrome1
    if dcm.PhotometricInterpretation == 'MONOCHROME1':
        img = img.max() - img
    # normalizo al rango 0-1
    img     = (img - img.min()) / (img.max() - img.min() + 1e-6)
    # convierto a imagen pil de 3 canales
    img_pil = Image.fromarray((np.stack([img, img, img], axis=2) * 255).astype(np.uint8))
    # redimensiono a 224 para gradcam
    img_224 = np.array(img_pil.resize((224, 224))) / 255.0
    # preparo el tensor para el modelo
    tensor  = transform_val(img_pil).unsqueeze(0).to(DEVICE)
    # genero el mapa de activación
    mapa    = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(0)])[0]
    # extraigo el bbox del mapa
    bbox    = extraer_bbox(mapa)
    # muestro la imagen original
    axes[idx, 0].imshow(img_224, cmap='gray')
    axes[idx, 0].set_title(f'original — {pid[:8]}')
    axes[idx, 0].axis('off')
    # superpongo el mapa de calor sobre la imagen
    overlay = show_cam_on_image(img_224.astype(np.float32), mapa, use_rgb=True)
    axes[idx, 1].imshow(overlay)
    # dibuja el bbox si existe
    if bbox:
        x, y, w, h = bbox
        # escalo el bbox a 224 para la visualización
        escala = 224 / 1024
        rect = patches.Rectangle(
            (x * escala, y * escala), w * escala, h * escala,
            linewidth=2, edgecolor='red', facecolor='none'
        )
        axes[idx, 1].add_patch(rect)
    axes[idx, 1].set_title('GradCAM + bbox')
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()

# Inferencia sobre test y submission con GradCAM

In [ ]:
# evaluo radiografías de prueba, calculo probabilidades de neumonía y delimitando
# coordenadas de recuadros mediante la técnica GradCAM

# obtengo los ids de test desde la carpeta de dicoms
ids_test   = [f.replace('.dcm', '') for f in os.listdir(RUTA_TEST)]
# lista para almacenar los resultados
resultados = []
# activa el modo evaluación del modelo
modelo.eval()

for pid in ids_test:
    # leo el dicom de test directamente
    dcm    = pydicom.dcmread(f'{RUTA_TEST}/{pid}.dcm')
    # extraigo el array de píxeles
    img    = dcm.pixel_array.astype(np.float32)
    # invierto si es monochrome1
    if dcm.PhotometricInterpretation == 'MONOCHROME1':
        img = img.max() - img
    # normalizo al rango 0-1
    img     = (img - img.min()) / (img.max() - img.min() + 1e-6)
    # convierto a imagen pil de 3 canales
    img_pil = Image.fromarray((np.stack([img, img, img], axis=2) * 255).astype(np.uint8))
    # preparo el tensor para el modelo
    tensor  = transform_val(img_pil).unsqueeze(0).to(DEVICE)
    # calculo la probabilidad de neumonía
    prob    = torch.sigmoid(modelo(tensor).squeeze()).item()

    if prob > 0.5:
        # genero el mapa de activación para localizar el foco
        mapa = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(0)])[0]
        # extraigo el bbox del mapa
        bbox = extraer_bbox(mapa)
        if bbox:
            # construyo el string con confianza y coordenadas
            x, y, w, h = bbox
            pred_str = f'{prob:.4f} {x} {y} {w} {h}'
        else:
            # uso bbox completa si gradcam no detecta zona activa
            pred_str = f'{prob:.4f} 0 0 1024 1024'
    else:
        # cadena vacía si no hay neumonía
        pred_str = ''

    # agrega el resultado a la lista
    resultados.append({'patientId': pid, 'PredictionString': pred_str})

# convierte la lista en dataframe
df_sub = pd.DataFrame(resultados)
# guarda la submission en disco
df_sub.to_csv(f'{RUTA_OUTPUTS}/submission_fase21_gradcam.csv', index=False)
# muestra el tamaño y las primeras filas
print(f'submission generada: {len(df_sub)} filas')
df_sub.head()